In [ ]:
!pip install streamlit

We need to upload the files in Google drive
The path is - /content/drive/MyDrive/5. Learning AI/0.0 FDE role Prep - Interview kickstart/Projects/Week 0/Question - Week 0_Mini Project/Datasets


In [ ]:
!wget -q -O - ipv4.icanhazip.com

34.21.134.237


In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Week 0 Assignment", layout="wide")
st.title("Hello world program using streamlit and Gemini")
st.write("Ask your question below and get answer from a LLM")

user_query = st.text_input("Enter your question")

import os
import pandas as pd
from google import genai

# 1. Initialize client
# api_key = userdata.get('GEMINI_API_KEY') # Modified to get from environment variable
api_key = os.environ.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 2. Folder path
folder_path = '/content/drive/MyDrive/5. Learning AI/0.0 FDE role Prep - Interview kickstart/Projects/Week 0/Question - Week 0_Mini Project/Datasets'

# 3. Read and format all CSVs into knowledge context
dataset_context = ""

for filename in sorted(os.listdir(folder_path)):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)
        dataset_context += f"{df.to_csv(index=False)}\n"

# 4. Prompt Gemini
prompt = f"""
You are a precise data assistant. Your job is to answer user questions using ONLY the facts and values present in the provided CSV datasets below.

### STRICT RULES & CONSTRAINTS:
1. ONLY USE PROVIDED DATA: Answer solely using the exact rows, columns, numbers, and facts found in the context.
2. NO SPECULATION / HALLUCINATION: If the provided data does not contain the answer, or if the question requires external context not present in the tables, you must state: "Based on the provided data, I cannot answer this question because the information is not present in the dataset."
3. CITE SPECIFICS: Reference the specific dataset/table name and column values whenever possible to support your answer.

---
### PROVIDED DATASETS:
{dataset_context}

---
### USER QUESTION:
{user_query}
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)
#print(dataset_context)
st.write(response.text)

Overwriting app.py


In [ ]:
import subprocess
import time
import os
from google.colab import userdata # Added this import

# 1. Kill any existing instances
subprocess.run(["pkill", "-9", "streamlit"], stderr=subprocess.DEVNULL)

# Get the API key from Colab's userdata secrets and set it in the current environment.
# This ensures it's available for the Streamlit subprocess.
api_key = userdata.get('GEMINI_API_KEY') # Changed this line to use userdata.get()

if api_key is None:
    raise ValueError("GEMINI_API_KEY is not set in Colab secrets. Please add it to 'Secrets' in the left sidebar.")

os.environ['GEMINI_API_KEY'] = api_key # Explicitly set it in the current environment

# 2. Spawn Streamlit in an independent, fully detached background process
cmd = [
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--browser.gatherUsageStats", "false"
]

# Create a copy of the current environment variables to pass to the subprocess.
# os.environ.copy() will now include the GEMINI_API_KEY we just set.
env = os.environ.copy()

subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)

# 3. Wait briefly for the server to bind to port 8501
time.sleep(3)
print("Streamlit is running in the background!")

Streamlit is running in the background!


In [ ]:
import time
from google.colab import output

time.sleep(3)
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>